# Additional Functionality in Keras
Outside of building simple networks, there is a lot of additional functionality that can add predictive power, stability, and optimizations to our networks. In this lecture we will talk about:
 - Saving and Loading Models
 - Batch Normalization
 - Dropout
 - Early Stopping
 - Hyperparameter GridSearch

### Setting Up Our Data

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

seed = 10
np.random.seed(seed)

# Sonar dataset is used to classify mines vs rocks
dataframe = pd.read_csv("sonar.csv", header=None)
dataset = dataframe.values
# split into inputs (X) and outputs (Y) variables
X = dataset[:,0:60].astype(float)
Y = dataset[:,60]
# Encode our data for classification
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)

X_train, X_test, y_train, y_test = train_test_split(X, encoded_Y, test_size=0.33, random_state=22)

In [ ]:
print(X[:1])
print(Y)
print(encoded_Y)

In [3]:
def sonar_model(optimizer='adam', init='normal'):
    model = Sequential()
    model.add(Dense(120, input_dim=60, kernel_initializer=init, activation='relu'))
    model.add(Dense(60, kernel_initializer=init, activation='relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

## Saving Keras Models
Typically we aren't creating and using a model all in the same session, meaning we'll need a way to save out our models. The current standard for saving the weights of a neural network is the hd5 file format, the keras model itself can actually be saved out via json.

This means there are two steps to saving/loading keras models:
 1. Saving out the weights of the model in an hd5 file
 2. Saving out the model structure using json

### Saving Our Models
To save our models we'll use one function:
 - model.save()

In [ ]:
model = sonar_model()
model.fit(X_train, y_train)

In [ ]:
model.save("sonar_model")

Let's take a quick look at the json file we saved out.

### Loading In a Saved Model
To load a saved model we need to simply reference the folder with the saved attributes

In [6]:
from tensorflow.keras.models import load_model
reconstructed_model = load_model("sonar_model")

In [ ]:
reconstructed_model.predict(X_test[:2])

### In-Class Exercise
Validate that the loaded_model performs as expected.

In [8]:
# Space for in-class exercise

----

## Random Seeds with Tensorflow
Many operations in Tensorflow, and data science in general, are driven by randomness. For example, the initialization for our network weights are generated through a random number generator to avoid any inherent bias in their selection.

While randomness is required for developing models, we also need the ability to control randomness fo consistency. We do this through a concept called a **random seed**. Seeds provide the intiialization state for the psuedo random number generators used in packages like TensorFlow and Numpy.

### Setting Random Seeds
 - Numpy:
   1. Create a rng generator `rng = np.random.default_rng({seed})`
   2. Use the rng generator to create random values -> `rng.random()`
 - TensorFlow:
   1. Set the rando seed through `tf.random.set_seed({seed})`

#### Numpy Example:

In [ ]:
import numpy as np

rand_1 = np.random.random(5)
rand_2 = np.random.random(5)

print("Without a random number generator:")
print(rand_1)
print(rand_2)

rng = np.random.default_rng(10)
rand_3 = rng.random(5)
rng = np.random.default_rng(10)
rand_4 = rng.random(5)

print("\nWith a random number generator:")
print(rand_3)
print(rand_4)

#### TensorFlow Example:

In [ ]:
import tensorflow as tf

rand_1 = tf.random.normal((5,))
rand_2 = tf.random.normal((5,))

print("Without a random number generator:")
print(rand_1)
print(rand_2)

tf.random.set_seed(10)
rand_3 = tf.random.normal((5,))
tf.random.set_seed(10)
rand_4 = tf.random.normal((5,))

print("\nWith a random number generator:")
print(rand_3)
print(rand_4)

-----
## Batch Normalization
Batch normalization is the process of normalizing the inputs/outputs at given layers of the network. The concept here is that by normalizing at each layer we can improve the training performance our models, as each layer is working on standardized inputs.

Good summary of the benefits of [Batch Normalization](https://towardsdatascience.com/batch-norm-explained-visually-how-it-works-and-why-neural-networks-need-it-b18919692739)

In our instance below I've updated the *momentum* parameter to better fit our data, as we have relatively small amount of data with limited variation, for this reason we don't want the training data to have too large an impact on our trained parameters.

In [ ]:
lst = []
lst.append('BN')
lst.append('Dense')
lst

In [12]:
from tensorflow.keras.layers import BatchNormalization, Activation

def sonar_batch_model(optimizer='adam', init='normal'):
    model = Sequential()
    model.add(Dense(120, input_dim=60, kernel_initializer=init, name='input_layer'))
    model.add(BatchNormalization(momentum=0.5, epsilon=0.001, name='bn_input_layer'))
    model.add(Activation('relu'))
    model.add(Dense(60, kernel_initializer=init))
    model.add(BatchNormalization(momentum=0.5, epsilon=0.001))
    model.add(Activation('relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

In [ ]:
model = sonar_batch_model()
model.summary()

### In-Class Exercise
Compare the performance of our initial sonar_model with our batch normalized model.

In [14]:
# Space for in-class exercise


### How Batch Normalization Works At the Layer Level

**Batch Normalization**, as described in it's [original paper](https://arxiv.org/pdf/1502.03167v3.pdf), is a process where each batch of training data is normalized using the mean and standard deviation of the batches of data, using the following calculations:

<img src=https://miro.medium.com/max/2000/1*VsN_9_AN2ji8hCZYSTTV0w.png></img>
*<a href=https://towardsdatascience.com/batch-norm-explained-visually-how-it-works-and-why-neural-networks-need-it-b18919692739>image source</a>*

This means that when this model makes predictions, it will used trained values for the gamma ($\gamma$), beta ($\beta$), mean, and std. We can see these weights (or trained values) by looking at the weights of our batch normalization layer.

**batch_model.layers\[bn_layer\].get_weights() - with params at these indexes:**
 1. gamma
 2. beta
 3. running mean
 4. running std


## Dropout
In addition to **Batch Normalization** another technique for improving our NN is called **Dropout**. This technique works by actually filtering out certain nodes during the training process, meaning during each batch of training we deactivate/drop certain nodes. This benefits our model by preventing overfitting, *i.e. relying to heavily on certain paths through our network*.

As seen in the image below, different pathways will be dropped (deactivated) changing the possible pathways through the network. This is typically done during each layer update, menaing that it may happen at the batch or epoch level depending on the implementation.
<img src='https://s3-ap-south-1.amazonaws.com/av-blog-media/wp-content/uploads/2018/04/1IrdJ5PghD9YoOyVAQ73MJw.gif'/>

There are a couple of different reasons why adding dropout layers to your network may be beneficial. However, in general it is seen as a way to force a more robust modeling of the data. Essentially by constantly shifting the landscape a bit it ensures that layers won't co-adapt while also providing a means for the network to learn multiple different pathways through the network to sucessfully process data. This also helps prevent an abundance of dead neurons (neurons never used/updated) by constantly shifting how each neuron is calculated.

Some basic rules for Dropout are:
 - The larger the network, the more beneficial it is to use Dropout
 - Typically 20% dropout is a good starting place, but can be increased/decrease depending on testing
 - It is recommended to use Dropout on input and hidden layers, but this is dependent on the number of features being used.

In [15]:
from tensorflow.keras.layers import Dropout

def sonar_input_dropout_model(optimizer='adam', init='normal'):
    model = Sequential()
    model.add(Dropout(0.2, input_shape=(60,)))
    model.add(Dense(120, kernel_initializer=init, activation='relu'))
    model.add(Dropout(0.2,))
    model.add(Dense(60, kernel_initializer=init, activation='relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model


In [16]:
def sonar_dropout_model(optimizer='adam', init='normal'):
    model = Sequential()
    model.add(Dense(120, input_dim=60, kernel_initializer=init, activation='relu'))
    model.add(Dropout(0.2,))
    model.add(Dense(60, kernel_initializer=init, activation='relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model


If you're curious about if and when you can intermix BatchNorm and Dropout this paper details a study to answer those questions - [Dropout vs. batch normalization: an empirical study of their impact to deep learning](https://wrlc-gwu.primo.exlibrisgroup.com/permalink/01WRLC_GWA/15suu1b/cdi_crossref_primary_10_1007_s11042_019_08453_9)

In [ ]:
sonar_input_dropout_model = sonar_input_dropout_model()
sonar_input_dropout_model.fit(X_train, y_train, epochs=10, batch_size=10)

In [ ]:
sonar_dropout_model = sonar_dropout_model()
sonar_dropout_model.fit(X_train, y_train, epochs=10, batch_size=10)

----
## Early Stopping
Early stopping is a technique that can end a training cycle when certain criteria are met. The base premise being that if certain model metrics aren't improving, it may indicate that we can stop early as the model can't improve.

### Configuring
```python
es = EarlyStopping(
    monitor='val_loss',
    mode='min',
    verbose=1,
    patience=50,
    min_delta,
    baseline,
)
```
 - monitor - The monitor parameter indicates the metric used to determine if the model training should be cut early
 - mode {'auto', 'min', 'max'} - Indicates whether we want to minimize/maximize the monitored metric
 - patience - Number of epochs with no improvement after which training will be stopped
 - min_delta - The minimum change in the monitored metric to be considered an improvement, and thus resetting the patience count
 - baseline - A baseline value we must pass or training will be stopped early

In [19]:
from tensorflow.keras.optimizers import Adam

def sonar_model(init='normal'):
    model = Sequential()
    model.add(Dense(120, input_dim=60, kernel_initializer=init, activation='relu'))
    model.add(Dense(40, kernel_initializer=init, activation='relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:
tf.random.set_seed(10)
model = sonar_model()
model.fit(X_train, y_train, epochs=25, batch_size=5, validation_data=(X_test, y_test), verbose=2)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(10)
es = EarlyStopping(monitor='val_accuracy', verbose=1, patience=5, min_delta=0.1)
model = sonar_model()
model.fit(X_train, y_train, epochs=25, batch_size=5, validation_data=(X_test, y_test), verbose=2, callbacks=[es])

----
## Learning Rate Schedule
Similar to **early stopping**, there are other functions that can modify our model training. **Learning Rate Scheduling** is a process where we can adjust the learning rate of a model over a period of time.

Most **Learning Rate Scheudles** work on a decay principle:
 - CosineDecay: A LearningRateSchedule that uses a cosine decay schedule.
 - CosineDecayRestarts: A LearningRateSchedule that uses a cosine decay schedule with restarts.
 - ExponentialDecay: A LearningRateSchedule that uses an exponential decay schedule.
 - InverseTimeDecay: A LearningRateSchedule that uses an inverse time decay schedule.
 - LearningRateSchedule: The learning rate schedule base class.
 - PiecewiseConstantDecay: A LearningRateSchedule that uses a piecewise constant decay schedule.
 - PolynomialDecay: A LearningRateSchedule that uses a polynomial decay schedule.

**Learning Rate Schedule Parameters**:
 - initial_learning_rate: The initial value for the learning rate.
 - decay_steps: How many steps or times we decay the learning rate.
 - decay_rate: The rate of decay, $new_lr = old_lr * decay_rate$.
 
```python
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-2,
    decay_steps=10000,
    decay_rate=0.9)
optimizer = keras.optimizers.SGD(learning_rate=lr_schedule)
model.compile(loss='cross_entropy', optimizer=optimizer, metrics=['accuracy'])
```

***Note:*** *A training step is one gradient update. In one step __batch_size__ examples are processed.*

In [22]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import PiecewiseConstantDecay

def sonar_model(init='normal'):
    model = Sequential()
    model.add(Dense(120, input_dim=60, kernel_initializer=init, activation='relu'))
    model.add(Dense(40, kernel_initializer=init, activation='relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    
    boundaries = [100, 200]
    values = [1.0, 0.5, 0.1]
    lr_schedule = PiecewiseConstantDecay(boundaries, values)
    
    optimizer = Adam(learning_rate=lr_schedule)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [ ]:
model.fit(X_train, y_train, epochs=25, batch_size=5, validation_data=(X_test, y_test), verbose=2)

### In Class
Combine two or more of our optimization techniques that we've reviewed to create an improved model.

In [ ]:
# Space for work


----
## Hyperparameter Search
A hyperparameter is a value that is used to define and/or configure a model. They are distinctly different from a model parameter, as they are not trainable values and are typically defined prior to model training/testing.

Examples of hyperparameters:
 - Number of hidden layers
 - Size of layers
 - Weight initialization
 - etc.
 
A *hyperparameter search* is an automated way to evaluate or search for the best hyperparameter set for a given problem

----

### Keras-Tuner
Keras tuner is one approach for hyperparameter tuning with keras. It works by defining hyperparameters as objects that can take on dynamic values, then running multiple experiments or runs of the model to explore the hyperparameter space.

In [ ]:
!pip install keras_tuner

### Keras Tuner
With Keras Tuner variables are defined that can take on a range of of actual values. These variables are then used when defining hyperparameters, setting up the hyperparameter search.

```python
hidden_layer_num_nodes = hp.Int('hyper_param_name', min_value=X, max_value=X, step=X)
hidden_layer = Dense(hidden_layer_num_nodes)
```

Types of KT hyperparams:
 - Boolean
 - Choice
 - Fixed
 - Float
 - Int

***Note: Each hyperparam type has it's own type of configuraiton, but usually defines ranges for the hyperparam to take on.***

In [ ]:
import keras_tuner as kt

def sonar_model(hp):
    model = Sequential()
    hp_units_1 = hp.Int('units_1', min_value=60, max_value=120, step=20)
    model.add(Dense(hp_units_1, input_dim=60, activation='relu'))
    hp_units_2 = hp.Int('units_2', min_value=60, max_value=120, step=20)
    model.add(Dense(hp_units_2, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    optimizer = Adam(learning_rate=hp_learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

### Hyperband
From the Keras documentation:
>The Hyperband tuning algorithm uses adaptive resource allocation and early-stopping to quickly converge on a high-performing model. This is done using a sports championship style bracket. The algorithm trains a large number of models for a few epochs and carries forward only the top-performing half of models to the next round. Hyperband determines the number of models to train in a bracket by computing 1 + logfactor(max_epochs) and rounding it up to the nearest integer.

The following table describes the number of configurations n_i and the number of iterations they are run for r_i within each round of the Successive Halving innerloop for a particular value of (n,s).

*n = number of models - s = number of Succesive Halving - r = number of epochs*

| s=4 |  |s=3 | | s=2 | | s=1 | | s=0 | |
| --------- | --------- | --------- | --------- | --------- | --------- | --------- | --------- | --------- | --------- |
| **n_i** | **r_i \|** | **n_i** | **r_i \|** | **n_i** | **r_i \|** | **n_i** | **r_i \|** | **n_i** | **r_i \|** |
| 81 | 1 \| | 27 | 3 \| | 9 | 9 \| | 6 | 27 \| | 5 | 81 \| |
| 27 | 3 \| | 9 | 9 \| | 3 | 27 \| | 2 | 81 \| |  | 
| 9 | 9 \| | 3 | 27 \| | 1 | 81 \| |
| 3 | 27 \| | 1 | 81 \| |
|1 | 81 \| |

In [ ]:
tuner = kt.Hyperband(sonar_model,
                     objective='val_accuracy',
                     max_epochs=15,
                     factor=3,
                     seed=10,
                     directory='hpm',
                     project_name='sonar_model',
                     overwrite=True)
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)
tuner.search(X_train, y_train, epochs=50, batch_size=15, validation_split=0.2, callbacks=[stop_early])


In [ ]:
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal configuration found is:\n
\tHidden 1 Nodes: {best_hps.get('units_1')}\n
\tHidden 2 Nodes: {best_hps.get('units_2')}\n
\tLearning Rate: {best_hps.get('learning_rate')}.
""")



In [ ]:
model = tuner.hypermodel.build(best_hps)
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))


### Bayesian
> Bayesian approaches, in contrast to Hyperband, keep track of past evaluation results which they use to form a probabilistic model mapping hyperparameters to a probability of a score on the objective function:
$$ P(score|hyperparameters) $$

In [ ]:
b_tuner = kt.BayesianOptimization(
    sonar_model,
    objective='val_accuracy',
    max_trials=15,
    num_initial_points=2,
    alpha=0.0001,
    beta=2.6,
    seed=10,
    hyperparameters=None,
    directory='hpm',
    project_name='sonar_model',
    overwrite=True
)

# stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)
b_tuner.search(X_train, y_train, epochs=50, validation_split=0.2)#, callbacks=[stop_early])


In [ ]:
best_hps=b_tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal configuration found is:\n
\tHidden 1 Nodes: {best_hps.get('units_1')}\n
\tHidden 2 Nodes: {best_hps.get('units_2')}\n
\tLearning Rate: {best_hps.get('learning_rate')}.
""")


In [ ]:
model = b_tuner.hypermodel.build(best_hps)
history = model.fit(X_train, y_train, epochs=10, batch_size=15, validation_data=(X_test, y_test))


## In Class
Use a mixture of the optimization techniques we've reviewed to develop an optimal model

In [ ]:
# space for work
